In [10]:
# ============================================================
# Select and load a thermalized-state result
# ============================================================

from md_Helpers.simulation import get_or_make_thermalized_state

result = get_or_make_thermalized_state(
    n_fcc_cells=30,
    target_rho=0.755,
    kT=0.800,
    nsteps=1_000_000,
    seed=1,
    phase_name="randomization",
    overwrite=False,
)

print("\nRESULT")
print("=" * 100)
print("Created new:    ", result["created_new"])
print("State path:     ", result["paths"]["state_path"])
print("Log/metadata:   ", result["paths"]["log_path"])
print("State kind:     ", result["paths"]["state_kind"])
print("Phase name:     ", result["paths"]["phase_name"])

Loaded existing thermalized state:
/exp/e961/data/MDsims-data/pnichols/Thermalized_States_v3/FCC/n_cells_30/rho_0.755/kT_0.800/nsteps_1000000/seed_1/randomization.gsd

RESULT
Created new:     False
State path:      /exp/e961/data/MDsims-data/pnichols/Thermalized_States_v3/FCC/n_cells_30/rho_0.755/kT_0.800/nsteps_1000000/seed_1/randomization.gsd
Log/metadata:    /exp/e961/data/MDsims-data/pnichols/Thermalized_States_v3/FCC/n_cells_30/rho_0.755/kT_0.800/nsteps_1000000/seed_1/randomization_log.hdf5
State kind:      thermalized
Phase name:      randomization


In [11]:
from pathlib import Path

import h5py
import numpy as np

log_path = Path(result["paths"]["log_path"])


def readable(value):
    """Convert HDF5 and NumPy values into readable Python values."""
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, np.ndarray):
        return np.array2string(
            value,
            threshold=np.inf,
            linewidth=120,
        )

    return value


print("THERMALIZED-STATE METADATA")
print(f"File: {log_path}")

with h5py.File(log_path, "r") as hdf:
    if "metadata" not in hdf:
        raise KeyError(f"No /metadata group exists in {log_path}")

    metadata_groups = ["/metadata"]

    def find_groups(name, obj):
        if isinstance(obj, h5py.Group):
            metadata_groups.append(f"/metadata/{name}")

    hdf["metadata"].visititems(find_groups)

    for group_path in metadata_groups:
        group = hdf[group_path]

        print("\n")
        print("=" * 100)
        print(group_path)
        print("=" * 100)

        item_number = 1

        # Print attributes stored directly on this group
        if group.attrs:
            print("\nATTRIBUTES")

            for name in sorted(group.attrs):
                value = readable(group.attrs[name])

                print(f"\n[{item_number}] {name}")
                print(f"    storage: attribute")
                print(f"    type:    {type(value).__name__}")
                print(f"    value:   {value}")

                item_number += 1

        # Print datasets stored directly inside this group
        datasets = [
            (name, obj)
            for name, obj in group.items()
            if isinstance(obj, h5py.Dataset)
        ]

        if datasets:
            print("\nDATASETS")

            for name, dataset in sorted(datasets):
                value = readable(dataset[()])

                print(f"\n[{item_number}] {name}")
                print(f"    storage: dataset")
                print(f"    dtype:   {dataset.dtype}")
                print(f"    shape:   {dataset.shape}")
                print(f"    value:   {value}")

                item_number += 1

        if not group.attrs and not datasets:
            print("\n(empty container group)")

THERMALIZED-STATE METADATA
File: /exp/e961/data/MDsims-data/pnichols/Thermalized_States_v3/FCC/n_cells_30/rho_0.755/kT_0.800/nsteps_1000000/seed_1/randomization_log.hdf5


/metadata

(empty container group)


/metadata/classification

(empty container group)


/metadata/classification/phase_separation

ATTRIBUTES

[1] phase_separated
    storage: attribute
    type:    bool
    value:   False


/metadata/classification/phase_separation/voxel

ATTRIBUTES

[1] density_threshold
    storage: attribute
    type:    float
    value:   0.2

[2] low_density_fraction
    storage: attribute
    type:    float
    value:   0.0

[3] method
    storage: attribute
    type:    str
    value:   voxel_low_density_fraction

[4] nbins
    storage: attribute
    type:    int
    value:   12

[5] nbins_source
    storage: attribute
    type:    str
    value:   n_fcc_cells_rule

[6] phase_separated
    storage: attribute
    type:    bool
    value:   False

[7] updated_from_saved_gsd
    storage: attrib

In [15]:
# ============================================================
# Clean metadata in all existing thermalized-state logs
# ============================================================

from collections import Counter
from importlib import reload

from md_Helpers import metadata as metadata_helpers
from md_Helpers.paths import THERMALIZED_STATES_V3_ROOT

# Reload so a running notebook sees the updated helper code.
reload(metadata_helpers)

# First run: False previews without modifying anything.
# After reviewing the output, set this to True and rerun.
APPLY_CHANGES = False

reports = metadata_helpers.cleanup_thermalized_metadata_tree(
    root=THERMALIZED_STATES_V3_ROOT,
    dry_run=not APPLY_CHANGES,
)

if not reports:
    raise RuntimeError(
        f"No HDF5 files were found under {THERMALIZED_STATES_V3_ROOT}"
    )

counts = Counter(report["status"] for report in reports)
errors = [
    report
    for report in reports
    if report["status"] == "error"
]

print("=" * 100)
print("THERMALIZED METADATA CLEANUP")
print("=" * 100)
print("Root:          ", THERMALIZED_STATES_V3_ROOT)
print("Mode:          ", "APPLY CHANGES" if APPLY_CHANGES else "DRY RUN")
print("Files checked: ", len(reports))

for status, count in sorted(counts.items()):
    print(f"{status:15} {count}")

print("\nFILES WITH CHANGES")
print("=" * 100)

for report in reports:
    if report["status"] not in {"would_clean", "cleaned"}:
        continue

    print(
        f"\n{report['status'].upper()}: "
        f"{report['hdf5_path']}"
    )
    print(f"Fields: {report['removed_count']}")

    for item in report["removed"]:
        print(f"  - /{item['group']}/{item['attribute']}")

print("\nERRORS")
print("=" * 100)

if errors:
    for report in errors:
        print(f"\nERROR: {report['hdf5_path']}")
        print(f"       {report['error']}")
else:
    print("No errors found.")

if not APPLY_CHANGES:
    print("\nDRY RUN ONLY: no files were modified.")
    print("Set APPLY_CHANGES = True and rerun to perform the cleanup.")
else:
    print("\nCleanup applied.")

THERMALIZED METADATA CLEANUP
Root:           /exp/e961/data/MDsims-data/pnichols/Thermalized_States_v3
Mode:           DRY RUN
Files checked:  1172
already_clean   1172

FILES WITH CHANGES

ERRORS
No errors found.

DRY RUN ONLY: no files were modified.
Set APPLY_CHANGES = True and rerun to perform the cleanup.
